Load the objects imported from 01_load_data notebook

In [ ]:
# Define base path
base_path = "path/to/project"

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath("../"))

from config import base_path, output_base, gene_names, ja_experiment_id

experiment_id = os.path.basename(base_path)

In [ ]:
import spatialdata as sd
import rioxarray
import numpy as np
import matplotlib.pyplot as plt
import os

analysis_path = base_path.replace("/output/", "/analysis/")
experiment_id = os.path.basename(base_path)

# Auto-detect regions
regions = sorted([
    d for d in os.listdir(base_path)
    if os.path.isdir(os.path.join(base_path, d)) and d.startswith("region_")
])
print(f"Found regions: {regions}")

sdata_dict = {}
for region in regions:
    sdata_dict[region] = sd.read_zarr(f"{analysis_path}/{region}.zarr")
    print(f"{region}: loaded")

For each region, generate a low-res image with pixel grid to help identify root sections

In [ ]:
overview_dir = f"{output_base}/plots/{experiment_id}/overviews"
os.makedirs(overview_dir, exist_ok=True)

scale = 16
channel = "PolyT"
z_layer = 3
mosaic = f"mosaic_{channel}_z{z_layer}"

for region in regions:
    polyt_lowres = sdata_dict[region][mosaic][:, ::scale, ::scale].isel(c=0).compute()

    vmin = float(np.percentile(polyt_lowres, 1))
    vmax = float(np.percentile(polyt_lowres, 99))

    fig, ax = plt.subplots(figsize=(12, 9))
    ax.imshow(polyt_lowres, cmap="gray", vmin=vmin, vmax=vmax)
    ax.grid(True, color='red', alpha=0.3, linewidth=0.5)
    ax.set_title(f"{region} - {channel} overview")
    plt.savefig(f"{overview_dir}/{region}_{channel}_overview.png", dpi=200)
    plt.close()
    print(f"{region}: overview saved")

Check one gene per moisaic to see which sections have good expression to subsequently crop individually

In [ ]:
#verify_gene = "Solyc04g081490"  # change to any gene you expect to be expressed

for region in regions:
    points_layer = [k for k in sdata_dict[region].points.keys()][0]
    df_r = sdata_dict[region][points_layer].compute()
    
    polyt_lowres = sdata_dict[region]["mosaic_PolyT_z3"][:, ::scale, ::scale].isel(c=0).compute()
    vmin = float(np.percentile(polyt_lowres, 1))
    vmax = float(np.percentile(polyt_lowres, 99))

    gene_subset = df_r[df_r["gene"] == verify_gene]

    fig, ax = plt.subplots(figsize=(12, 9))
    ax.imshow(polyt_lowres, cmap="gray", vmin=vmin, vmax=vmax)
    
    # Overlay transcripts scaled to low-res coordinates
    ax.scatter(
        gene_subset["x"] / scale,
        gene_subset["y"] / scale,
        s=0.5, c="crimson", alpha=0.7, rasterized=True
    )
    ax.set_title(f"{region} - {verify_gene} (n={len(gene_subset)})")
    ax.axis("off")
    plt.savefig(f"{overview_dir}/{region}_{verify_gene}_overview.png", dpi=200)
    plt.close()
    print(f"{region}: done")

Coordinates of sections within the overview images

In [ ]:
# xmin, xmax, ymin, ymax in pixel coordinates
roots = {
    "region_0": {
        "root1": (100, 700, 1300, 1700),
        "root2": (500, 1900, 300, 1500),
        "root3": (1800, 3600, 1200, 1750),
    },
    "region_1": {
        "root1": (200, 1300, 300, 1800),
        "root2": (600, 1200, 2500, 2850),
        "root3": (1500, 1650, 1700, 2400),
        "root4": (1050, 1250, 3550, 3800),
    },
    "region_2": {
        "root1": (200, 800, 150, 1300),
    },
    "region_3": {
        "root1": (150, 1450, 150, 1350),
    },
    "region_4": {
        "root1": (550, 950, 300, 750),
        "root2": (380, 1200, 1150, 1400),
    },
    "region_5": {
        "root1": (130, 1200, 200, 1100),
    },
    "region_6": {
        "root1": (1450, 2520, 250, 1600),
        "root2": (650, 1000, 2300, 2750),
    },
    "region_7": {
        "root1": (250, 500, 200, 850),
        "root2": (1000, 2250, 600, 1800),
        "root3": (2750, 3800, 2300, 2800),
    },
    "region_8": {
        "root1": (200, 650, 0, 810),
        "root2": (1800, 2250, 100, 700),
    },
    "region_9": {
        "root1": (450, 1750, 400, 620),
    },
}

Check one gene per root per region to see if crop is okay

In [ ]:
import harpy as hp
import spatialdata as sd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np
import os

verify_dir = f"{output_base}/plots/verify/{experiment_id}"
os.makedirs(verify_dir, exist_ok=True)

for region, region_roots in roots.items():
    print(f"Loading {region}...")
    sdata_r = sd.read_zarr(f"{analysis_path}/{region}.zarr")
    
    img_layer = [k for k in sdata_r.images.keys()][0]
    points_layer = [k for k in sdata_r.points.keys()][0]
    df_r = sdata_r[points_layer].compute()

    for root_name, (xmin, xmax, ymin, ymax) in region_roots.items():
        xmin_s, xmax_s, ymin_s, ymax_s = xmin*scale, xmax*scale, ymin*scale, ymax*scale

        crop = sdata_r[img_layer][:, ymin_s:ymax_s, xmin_s:xmax_s].isel(c=0).compute()
        norm = Normalize(vmin=float(np.percentile(crop, 1)), vmax=float(np.percentile(crop, 99)))
        del crop

        subset = df_r[(df_r["gene"] == verify_gene) &
                      (df_r["x"] > xmin_s) & (df_r["x"] < xmax_s) &
                      (df_r["y"] > ymin_s) & (df_r["y"] < ymax_s)]

        out_path = f"{verify_dir}/{region}_{root_name}_{verify_gene}.png"
        if os.path.exists(out_path):
            print(f"  {root_name}: skipping (already exists)")
            continue

        fig, ax = plt.subplots(figsize=(8, 8))
        hp.pl.plot_sdata(
            sdata_r,
            img_layer=img_layer,
            channel=0,
            crd=[xmin_s, xmax_s, ymin_s, ymax_s],
            to_coordinate_system="global",
            ax=ax,
            render_images_kwargs={"cmap": "gray", "norm": norm},
        )
        ax.scatter(subset["x"], subset["y"], s=1, c="crimson", alpha=0.8)
        ax.set_title(f"{region} - {root_name} - {verify_gene} (n={len(subset)})")

        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  {root_name}: done")

    del sdata_r, df_r
    print(f"{region}: complete")

In [ ]:
# Add gene names from file

import pandas as pd

# Load gene name mapping
gene_names = pd.read_csv(gene_names, encoding="latin-1")
print(gene_names.head())
print(f"Total genes in list: {len(gene_names)}")

# Clean solycid column , remove transcript info
gene_names["solycid"] = gene_names["solycid"].str[:14]

# Create a lookup dictionary: solycid -> gene_name
gene_name_dict = dict(zip(gene_names["solycid"], gene_names["gene_name"]))

# Helper function to get display name
def get_display_name(solyc_id):
    if solyc_id in gene_name_dict:
        return f"{solyc_id}_{gene_name_dict[solyc_id]}"
    return solyc_id  # fall back to solyc ID if not in list

# Test
print(get_display_name(verify_gene))

Run all genes for all roots of all regions for JA - group them per gene. Individual plots

In [ ]:
import harpy as hp
import spatialdata as sd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np
import os
import gc

scale = 16
for region, region_roots in roots.items():
    print(f"Loading {region}...")
    sdata_r = sd.read_zarr(f"{analysis_path}/{region}.zarr")
    
    img_layer = [k for k in sdata_r.images.keys()][0]
    points_layer = [k for k in sdata_r.points.keys()][0]
    df_r = sdata_r[points_layer].compute()
    panel_genes_r = [g for g in df_r["gene"].unique() if "blank" not in g.lower()]

    for root_name, (xmin, xmax, ymin, ymax) in region_roots.items():
        xmin_s, xmax_s, ymin_s, ymax_s = xmin*scale, xmax*scale, ymin*scale, ymax*scale

        crop = sdata_r[img_layer][:, ymin_s:ymax_s, xmin_s:xmax_s].isel(c=0).compute()
        norm = Normalize(vmin=float(np.percentile(crop, 1)), vmax=float(np.percentile(crop, 99)))
        del crop

        for gene in panel_genes_r:
            display_name = get_display_name(gene)
            plot_dir = f"{output_base}/plots/{experiment_id}/genes/{display_name}"
            os.makedirs(plot_dir, exist_ok=True)
            
            
            out_path = f"{plot_dir}/{display_name}_{region}_{root_name}.png"
            if os.path.exists(out_path):
                continue

            subset = df_r[(df_r["gene"] == gene) &
                          (df_r["x"] > xmin_s) & (df_r["x"] < xmax_s) &
                          (df_r["y"] > ymin_s) & (df_r["y"] < ymax_s)]

            fig, ax = plt.subplots(figsize=(8, 8))
            hp.pl.plot_sdata(
                sdata_r,
                img_layer=img_layer,
                channel=0,
                crd=[xmin_s, xmax_s, ymin_s, ymax_s],
                to_coordinate_system="global",
                ax=ax,
                render_images_kwargs={"cmap": "gray", "norm": norm},
            )
            ax.scatter(subset["x"], subset["y"], s=1, c="crimson", alpha=0.8)
            ax.set_title(f"{display_name} | {region} | {root_name} (n={len(subset)})")

            plt.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close('all')

    del sdata_r, df_r
    gc.collect()
    print(f"{region}: complete")

Plot all regions per gene together, optional

In [ ]:
import math

# Collect all region/root combinations
all_combos = [(region, root_name, coords) 
              for region, region_roots in roots.items() 
              for root_name, coords in region_roots.items()]

n_panels = len(all_combos)
n_cols = 3  # adjust to your preference
n_rows = math.ceil(n_panels / n_cols)

for gene in panel_genes_r:  # or use a master gene list
    out_path = f"{output_base}/plots/genes/{gene}/{gene}_all_regions.png"
    #if os.path.exists(out_path):
    #    continue
    display_name = get_display_name(gene)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 6*n_rows))
    axes = axes.flatten()
    
    # hide any unused panels
    for i in range(n_panels, len(axes)):
        axes[i].axis("off")

    for ax, (region, root_name, (xmin, xmax, ymin, ymax)) in zip(axes, all_combos):
        xmin_s, xmax_s, ymin_s, ymax_s = xmin*scale, xmax*scale, ymin*scale, ymax*scale

        sdata_r = sd.read_zarr(f"{output_base}/sdata_{region}.zarr")
        img_layer = [k for k in sdata_r.images.keys()][0]
        points_layer = [k for k in sdata_r.points.keys()][0]
        df_r = sdata_r[points_layer].compute()

        crop = sdata_r[img_layer][:, ymin_s:ymax_s, xmin_s:xmax_s].isel(c=0).compute()
        norm = Normalize(vmin=float(np.percentile(crop, 1)), vmax=float(np.percentile(crop, 99)))
        del crop

        subset = df_r[(df_r["gene"] == gene) &
                      (df_r["x"] > xmin_s) & (df_r["x"] < xmax_s) &
                      (df_r["y"] > ymin_s) & (df_r["y"] < ymax_s)]

        hp.pl.plot_sdata(
            sdata_r,
            img_layer=img_layer,
            channel=0,
            crd=[xmin_s, xmax_s, ymin_s, ymax_s],
            to_coordinate_system="global",
            ax=ax,
            render_images_kwargs={"cmap": "gray", "norm": norm},
        )
        ax.scatter(subset["x"], subset["y"], s=1, c="crimson", alpha=0.8)
        ax.set_title(f"{region}\n{root_name} (n={len(subset)})", fontsize=9)
        ax.axis("off")

        del sdata_r, df_r
        gc.collect()

    fig.suptitle(display_name, fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close('all')
    print(f"{gene}: combined plot saved")

For sequentially-imaged genes, load the mosaics and visualize per root per region

In [ ]:
import harpy as hp
import spatialdata as sd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np
import os
import gc

scale = 16
#seq_genes = ["gene1", "gene2"]

for gene in seq_genes:
    plot_dir = f"{output_base}/plots/genes/{gene}"
    os.makedirs(plot_dir, exist_ok=True)

    for region, region_roots in roots.items():
        sdata_r = sd.read_zarr(f"{output_base}/sdata_{region}.zarr")
        img_layer = [k for k in sdata_r.images.keys()][0]
        image_dir = f"{base_path}/{region}/images"

        # Load gene channel lazily
        gene_tif = rioxarray.open_rasterio(
            f"{image_dir}/mosaic_{gene}_z3.tif",
            chunks=(1, 4096, 4096), lock=False
        ).rename({"band": "c"})

        for root_name, (xmin, xmax, ymin, ymax) in region_roots.items():
            xmin_s, xmax_s, ymin_s, ymax_s = xmin*scale, xmax*scale, ymin*scale, ymax*scale

            out_path = f"{plot_dir}/{gene}_{region}_{root_name}.png"
            if os.path.exists(out_path):
                continue

            # PolyT crop
            polyt_crop = sdata_r[img_layer][:, ymin_s:ymax_s, xmin_s:xmax_s].isel(c=0).compute()
            polyt_norm = Normalize(vmin=float(np.percentile(polyt_crop, 1)), vmax=float(np.percentile(polyt_crop, 99)))
            del polyt_crop

            # Gene channel crop
            gene_crop = gene_tif.isel(c=0)[ymin_s:ymax_s, xmin_s:xmax_s].compute()
            gene_norm = Normalize(vmin=float(np.percentile(gene_crop, 1)), vmax=float(np.percentile(gene_crop, 99)))
            del gene_crop

            fig, axes = plt.subplots(1, 2, figsize=(12, 6))

            # PolyT crop for plotting
            polyt_plot = sdata_r[img_layer][:, ymin_s:ymax_s, xmin_s:xmax_s].isel(c=0).compute()
            polyt_norm = Normalize(vmin=float(np.percentile(polyt_plot, 1)), vmax=float(np.percentile(polyt_plot, 99)))

            # Gene channel crop
            gene_plot = gene_tif.isel(c=0)[ymin_s:ymax_s, xmin_s:xmax_s].compute()
            gene_norm = Normalize(vmin=float(np.percentile(gene_plot, 1)), vmax=float(np.percentile(gene_plot, 99)))

            fig, axes = plt.subplots(1, 2, figsize=(12, 6))

            axes[0].imshow(polyt_plot, cmap="gray", norm=polyt_norm, origin="upper")
            axes[0].set_title(channel)
            axes[0].axis("off")

            axes[1].imshow(gene_plot, cmap="hot", norm=gene_norm, origin="upper")
            axes[1].set_title(gene)
            axes[1].axis("off")

            fig.suptitle(f"{gene} | {region} | {root_name}", fontsize=12)
            plt.tight_layout()
            plt.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close('all')

        del sdata_r, gene_tif
        gc.collect()
        print(f"{gene} - {region}: done")

Stitch together representative roots to create a mock panel

In [ ]:
from PIL import Image
import math

def assemble_figure(image_paths, output_path, title, n_rows=2):
    images = [Image.open(p) for p in image_paths if os.path.exists(p)]
    if not images:
        print(f"No images found for {title}")
        return

    n_cols = math.ceil(len(images) / n_rows)
    
    w, h = images[0].size
    fig_w = w * n_cols
    fig_h = h * n_rows

    canvas = Image.new("RGB", (fig_w, fig_h), color=(255, 255, 255))
    for i, img in enumerate(images):
        row, col = divmod(i, n_cols)
        canvas.paste(img, (col * w, row * h))

    canvas.save(output_path, dpi=(150, 150))
    print(f"Saved: {output_path}")

# Define representative roots - same for all mock genes
representative_roots = [
    "region_R3_root1",
    "region_R4_root1", 
    "region_R4_root2",
    "region_R4_root3",
    "region_R5_root4",
    "region_R5_root5",
]

experiment_id = ja_experiment_id
for gene in gene_names["solycid"]:
    display_name = get_display_name(gene)
    summary_dir = f"{output_base}/plots/{experiment_id}/summaries"
    os.makedirs(summary_dir, exist_ok=True)

    # Mock — stitch representative roots
    mock_images = []
    for region_root in representative_roots:
        p = f"{output_base}/plots/{experiment_id}/genes/{display_name}/{display_name}_{region_root}.png"
        if os.path.exists(p):
            mock_images.append(p)
        else:
            print(f"Missing: {p}")

    # JA — use all_regions summary
    #ja_path = f"{output_base}/plots/{ja_experiment_id}/genes/{display_name}/{display_name}_all_regions.png"

    # Assemble mock panel
    if mock_images:
        assemble_figure(
            mock_images,
            f"{summary_dir}/{display_name}_JA_spatial.png",
            title=f"{display_name} - JA"
        )

    # Copy/open JA panel as-is
   # if os.path.exists(ja_path):
    #    ja_img = Image.open(ja_path)
    #    ja_img.save(f"{summary_dir}/{display_name}_JA_spatial.png")

    print(f"{display_name}: spatial panels done")

In [ ]:
def assemble_figure(image_paths, output_path, title, n_rows=2, target_height=1020):
    images = [Image.open(p) for p in image_paths if os.path.exists(p)]
    if not images:
        print(f"No images found for {title}")
        return

    # Resize all to same height
    resized = []
    for img in images:
        ratio = target_height / img.height
        new_w = int(img.width * ratio)
        resized.append(img.resize((new_w, target_height), Image.LANCZOS))

    n_cols = math.ceil(len(resized) / n_rows)
    
    # Build row by row, each row has its own total width
    rows = [resized[i*n_cols:(i+1)*n_cols] for i in range(n_rows)]
    row_widths = [sum(img.width for img in row) for row in rows]
    total_width = max(row_widths)
    total_height = target_height * n_rows

    canvas = Image.new("RGB", (total_width, total_height), color=(255, 255, 255))
    for r, row in enumerate(rows):
        x_offset = 0
        for img in row:
            canvas.paste(img, (x_offset, r * target_height))
            x_offset += img.width

    canvas.save(output_path, dpi=(150, 150))
    print(f"Saved: {output_path}")


# Define representative roots - same for all mock genes
representative_roots = [
    "region_R3_root1",
    "region_R4_root1", 
    "region_R4_root2",
    "region_R4_root3",
    "region_R5_root4",
    "region_R5_root5",
]

experiment_id = ja_experiment_id

for gene in gene_names["solycid"]:
    display_name = get_display_name(gene)
    summary_dir = f"{output_base}/plots/{experiment_id}/summaries"
    os.makedirs(summary_dir, exist_ok=True)

    # Mock — stitch representative roots
    mock_images = []
    for region_root in representative_roots:
        p = f"{output_base}/plots/{experiment_id}/genes/{display_name}/{display_name}_{region_root}.png"
        if os.path.exists(p):
            mock_images.append(p)
        else:
            print(f"Missing: {p}")

    # JA — use all_regions summary
    #ja_path = f"{output_base}/plots/{ja_experiment_id}/genes/{display_name}/{display_name}_all_regions.png"

    # Assemble mock panel
    if mock_images:
        assemble_figure(
            mock_images,
            f"{summary_dir}/{display_name}_JA_spatial.png",
            title=f"{display_name} - JA"
        )

    # Copy/open JA panel as-is
   # if os.path.exists(ja_path):
    #    ja_img = Image.open(ja_path)
    #    ja_img.save(f"{summary_dir}/{display_name}_JA_spatial.png")

    print(f"{display_name}: spatial panels done")